[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/byu-matrix-lab/torchlingo/blob/main/docs/docs/course/lecture-04-regex-refresher.ipynb)


# Lecture 4 reference: regular expressions for cleaning a corpus

**Reference. Ungraded. About 20 minutes.**

Assignment 4's cleaning steps are written as regular expressions: collapsing runs of whitespace,
stripping stray markup, keeping numbers and punctuation from splitting a segment. This notebook
is the pattern language on its own, on short made-up strings, so that the TMX cleaning activity
can spend its time on the data.

By the end you will be able to:

1. find literal text, any character, and one of a set of characters with `re.search` and
   `re.findall`;
2. tie a pattern to the start of a string, and say how many times a piece of it repeats;
3. pull parts of a match out with groups;
4. rewrite text with `re.sub`: collapse a run of whitespace, and strip markup tags without
   deleting the words between them.

These serve Lecture 4's outcome *Read a TMX file through `repr`, find a planted problem with code
rather than by eye, write a cleaning step as one function, and say which of the 16 steps fixes
which problem*.

Each exercise has a `# TODO:` where you replace `None` with your pattern. The cell stops until you
do; then it checks your answer, and if it is wrong, shows what your pattern matched. Run all
stops at the first exercise, which is expected.

<!-- GENERATED:purpose - change the torchlingo metadata, not this cell -->

**CS 479 course notebook** — offered alongside Lecture 4 (*Data Preparation for MT Training*).

In [ ]:
import re


---

## Literal characters

The simplest regex patterns match literal characters. The pattern `cat` will find the exact sequence of characters "c", "a", "t".

We'll use `re.search(pattern, text)`. This function scans the `text` looking for the *first* location where the `pattern` produces a match. It returns a special "match object" if found, otherwise it returns `None`. We use `r"..."` (raw strings) for patterns to avoid issues with backslashes.

**Exercise:** Find the word "fox" in the text below.

In [ ]:
text_to_search_1 = "The quick brown fox jumps over the lazy dog."

# TODO: a pattern that finds the word "fox".
# Hint: most characters in a pattern simply match themselves.
pattern_1 = None   # <- replace None with r"..."

if pattern_1 is None:
    raise NotImplementedError("Replace None with your pattern, then run this cell again.")

match_1 = re.search(pattern_1, text_to_search_1)
found = match_1.group() if match_1 else None
assert found == "fox", f"Expected to match 'fox'; your pattern matched {found!r}."

# .group() is the matched text; .start() and .end() are where it sits in the string
print(f"Found a match: '{found}' starting at index {match_1.start()} and ending at index {match_1.end()}")


---

## The dot and character sets

Metacharacters have special meanings in regex.

* `.` (Dot): Matches *any single character* (except newline `\n`).
* `[]` (Character Set): Matches *any one* of the characters inside the brackets.
    * Example: `[aeiou]` matches any vowel.
    * Ranges: `[a-z]` matches any lowercase letter, `[0-9]` matches any digit.

**Exercise 1:** Find any three-letter word starting with 'd', ending with 'g'.

In [ ]:
text_to_search_2 = "The quick brown fox jumps over the lazy dog digging a den."

# TODO: a pattern for 'd', then any single character, then 'g'.
# Hint: one metacharacter stands for any character at all.
pattern_2a = None   # <- replace None with r"..."

if pattern_2a is None:
    raise NotImplementedError("Replace None with your pattern, then run this cell again.")

match_2a = re.search(pattern_2a, text_to_search_2)
found = match_2a.group() if match_2a else None
assert found == "dog", f"Expected to match 'dog'; your pattern matched {found!r}."
print(f"Found a match using '.': '{found}' at index {match_2a.start()}")


**Exercise 2:** Find any occurrence of 'b', 'f', or 'd' followed immediately by 'o'. Use `re.findall()` which returns a list of *all* non-overlapping matches.


In [ ]:
# The same text as the exercise above (text_to_search_2)

# TODO: a pattern for b, f or d followed immediately by o.
# Hint: a character set matches exactly one character, whichever of its members is there.
pattern_2b = None   # <- replace None with r"..."

if pattern_2b is None:
    raise NotImplementedError("Replace None with your pattern, then run this cell again.")

matches_2b = re.findall(pattern_2b, text_to_search_2)
assert matches_2b == ["fo", "do"], f"Expected ['fo', 'do']; your pattern found {matches_2b}."
print(f"Found matches using '[]': {matches_2b}")


---

## Anchors and shorthands

* `^` (Caret): Matches the *start* of the string (or line in multi-line mode).
* `$` (Dollar): Matches the *end* of the string (or line in multi-line mode).
* `\d`: Matches any *digit* (equivalent to `[0-9]`).
* `\w`: Matches any *word character* (letters `[a-zA-Z]`, digits `[0-9]`, and underscore `_`).
* `\s`: Matches any *whitespace character* (space ` `, tab `\t`, newline `\n`, etc.).

**Exercise 1:** Check if the text *starts* with the word "The".

In [ ]:
text_to_search_3 = "The answer is 42."

# TODO: a pattern that matches "The" and the space after it, only at the start of the string.
# Hint: an anchor matches a position, not a character.
start_pattern = None   # <- replace None with r"..."

if start_pattern is None:
    raise NotImplementedError("Replace None with your pattern, then run this cell again.")

match_start = re.search(start_pattern, text_to_search_3)
found = match_start.group() if match_start else None
assert found == "The ", f"Expected to match 'The '; your pattern matched {found!r}."
assert re.search(start_pattern, "Is The answer 42?") is None, (
    "Your pattern also matches 'The' in the middle of a string; tie it to the start."
)
print(f"The text starts with 'The'. Match: '{found}'")


**Exercise 2:** Find the first sequence of exactly two digits.


In [ ]:
# The same text as the exercise above (text_to_search_3)

# TODO: a pattern for exactly two digits in a row.
# Hint: a shorthand matches one digit, whichever digit it is.
digit_pattern = None   # <- replace None with r"..."

if digit_pattern is None:
    raise NotImplementedError("Replace None with your pattern, then run this cell again.")

match_digits = re.search(digit_pattern, text_to_search_3)
found = match_digits.group() if match_digits else None
assert found == "42", f"Expected to match '42'; your pattern matched {found!r}."
print(f"Found two digits: '{found}'")


---

## Quantifiers

Quantifiers specify *how many* times the preceding character, group, or character set should occur.

* `*`: Zero or more times. (`go*gle` matches `ggle`, `google`, `gooogle`)
* `+`: One or more times. (`go+gle` matches `google`, `gooogle`, but not `ggle`)
* `?`: Zero or one time (makes it optional). (`colou?r` matches `color`, `colour`)
* `{n}`: Exactly `n` times. (`\d{3}` matches exactly 3 digits)
* `{n,}`: `n` or more times. (`\d{2,}` matches 2 or more digits)
* `{n,m}`: Between `n` and `m` times (inclusive). (`\d{2,4}` matches 2, 3, or 4 digits)

**Exercise:** Find all sequences of 3 or more digits in the text.

In [ ]:
text_to_search_4 = "Item 1 costs $99. Item 2 costs $1500. Item 3 is $7."

# TODO: a pattern for every run of three or more digits, used with re.findall.
# Hint: a quantifier in braces can set a minimum and leave the maximum open.
digit_seq_pattern = None   # <- replace None with r"..."

if digit_seq_pattern is None:
    raise NotImplementedError("Replace None with your pattern, then run this cell again.")

matches_digits = re.findall(digit_seq_pattern, text_to_search_4)
assert matches_digits == ["1500"], f"Expected ['1500']; your pattern found {matches_digits}."
print(f"Found digit sequences (3+ digits long): {matches_digits}")


---

## Grouping

Parentheses `()` create groups within your regex pattern. This is useful for:

1.  Applying quantifiers to multiple characters (e.g., `(ha)+` matches `ha`, `haha`, `hahaha`).
2.  Capturing parts of the match separately. `match.group(0)` (or `match.group()`) is the whole match, `match.group(1)` is the content of the first `()`, `match.group(2)` is the second, etc.

**Exercise:** Extract the protocol (http/https) and the domain name from URLs.

In [ ]:
text_to_search_5 = "Visit us at http://example.com or https://test-site.org today!"

# TODO: a pattern with two groups, the protocol (http or https) and the domain after "://".
# Hint: findall hands back only what the parentheses captured, one tuple per match, so put
# parentheses around exactly the two parts you want.
url_pattern = None   # <- replace None with r"..."

if url_pattern is None:
    raise NotImplementedError("Replace None with your pattern, then run this cell again.")

url_matches = re.findall(url_pattern, text_to_search_5)
wanted = [("http", "example.com"), ("https", "test-site.org")]
assert url_matches == wanted, f"Expected {wanted}; your pattern found {url_matches}."

print("Found URL parts (protocol, domain):")
for protocol, domain in url_matches:
    print(f" - Protocol: {protocol}, Domain: {domain}")


---

## Find and replace

`re.sub(pattern, replacement, text)` replaces every match of `pattern` in `text` with
`replacement` and returns the new string. A cleaning step that rewrites a segment is usually one
`re.sub`.

**Exercise 1:** Collapse every run of whitespace (spaces, tabs, line breaks) to a single space.

In [ ]:
text_to_clean_1 = "Study \t the\n\nword.   Then  pray."

# TODO: a pattern matching one whole run of whitespace, of any kind.
# Hint: if one match swallows the entire run, a single replacement leaves a single space.
space_pattern = None   # <- replace None with r"..."

if space_pattern is None:
    raise NotImplementedError("Replace None with your pattern, then run this cell again.")

collapsed = re.sub(space_pattern, " ", text_to_clean_1)
assert collapsed == "Study the word. Then pray.", (
    f"Expected 'Study the word. Then pray.' but got {collapsed!r}."
)
print(f"Collapsed: {collapsed!r}")


**Exercise 2:** Strip the markup tags and keep the words between them. The replacement is the
empty string, so whatever your pattern matches is deleted.

In [ ]:
text_to_clean_2 = "Press <b>Save</b>, then <i>Close</i>."

# TODO: a pattern matching one tag, opening or closing.
# Hint: a tag ends at the first ">" after its "<"; a pattern that runs on to a later ">"
# deletes the words between two tags as well.
tag_pattern = None   # <- replace None with r"..."

if tag_pattern is None:
    raise NotImplementedError("Replace None with your pattern, then run this cell again.")

stripped = re.sub(tag_pattern, "", text_to_clean_2)
assert stripped == "Press Save, then Close.", (
    f"Expected 'Press Save, then Close.' but got {stripped!r}; "
    f"your pattern matched {re.findall(tag_pattern, text_to_clean_2)}."
)
print(f"Stripped: {stripped!r}")


---

## What's next

The Lecture 4 TMX cleaning activity puts these patterns to work on a real file format.
[regex101.com](https://regex101.com/) shows what a pattern matches while you build it; choose its
Python flavor.